# Working with your data
GeoST offer various functions to read and parse subsurface data to GeoST objects. Generally
speaking, data can either be loaded from (local) files or is requested from a service like
the BRO REST-API. Either way, data coming from multiple sources and file formats are support. 

## Reading survey data
By survey data we refer to measurements (i.e. raw data) of the subsurface. These can comprise
boreholes, CPTs, Well logs, Seismic or EM lines and others (see [Data structures](./data_structures.ipynb#data-columns) for a more detailed description). In any case, the data is parsed to a [`geost.Collection`](../api_reference/collection.rst). The tables below list the currently supported data sources, associated reader functions and resulting GeoST objects.

| File format/data service | Read function  | Returned GeoST object | Description  |
| ------------------------ | -------------- | --------------------- | -----------  |    
| BHR-G | [`read_bhrg`](../api_reference/generated/geost.read_bhrg.rst) | [`Collection`](../api_reference/collection.rst) | (BRO) Geological boreholes from xml |
| BHR-GT | [`read_bhrgt`](../api_reference/generated/geost.read_bhrgt.rst) | [`Collection`](../api_reference/collection.rst) | (BRO) Geotechnical boreholes from xml |
| BHR-GT-samples | [`read_bhrgt_samples`](../api_reference/generated/geost.read_bhrgt_samples.rst) | [`Collection`](../api_reference/collection.rst) | (BRO) Geotechnical boreholes - grainsize samples from xml |
| BHR-P | [`read_bhrp`](../api_reference/generated/geost.read_bhrp.rst) | [`Collection`](../api_reference/collection.rst) | (BRO) Pedological boreholes from xml |
| CPT | [`read_cpt`](../api_reference/generated/geost.read_cpt.rst) [`read_gef_cpts`](../api_reference/generated/geost.read_gef_cpts.rst) |  [`Collection`](../api_reference/collection.rst) | (BRO) Cone Penetration Tests from xml or gef |
| SFR | [`read_sfr`](../api_reference/generated/geost.read_sfr.rst) | [`Collection`](../api_reference/collection.rst) | (BRO) Pedological soilprofile descriptions from xml |
| BRO REST-API | [`bro_api_read`](../api_reference/generated/geost.bro_api_read.rst) | [`Collection`](../api_reference/collection.rst) or  [`Collection`](../api_reference/collection.rst) | BRO BHR-G, BHR-GT, BHR-GT-samples, BHR-P, CPT or SFR objects |
| Parquet or csv | [`read_borehole_table`](../api_reference/generated/geost.read_borehole_table.rst) [`read_cpt_table`](../api_reference/generated/geost.read_cpt_table.rst) | [`Collection`](../api_reference/collection.rst) or [`DataFrame`](https://pandas.pydata.org/docs/reference/frame.html) | Survey data stored as a table. Result of `to_parquet` or `to_csv` export methods.
| NLOG excel export | [`read_nlog_cores`](../api_reference/generated/geost.read_nlog_cores.rst) | [`Collection`](../api_reference/collection.rst) or [`DataFrame`](https://pandas.pydata.org/docs/reference/frame.html) | Reader for NLOG deep cores, see [here](https://www.nlog.nl/boringen) |
| UU LLG cores | [`read_uullg_tables`](../api_reference/generated/geost.read_uullg_tables.rst) | [`Collection`](../api_reference/collection.rst) or [`DataFrame`](https://pandas.pydata.org/docs/reference/frame.html) | Reader for csv distribution of Utrecht University student boreholes |
| BORIS XML | [`read_xml_boris`](../api_reference/generated/geost.read_xml_boris.rst) | [`Collection`](../api_reference/collection.rst) or [`DataFrame`](https://pandas.pydata.org/docs/reference/frame.html) | Reader for XML exports of the BORIS borehole description software |

### Reading data from the BRO REST-API
Subsurface data is widely available in the Netherlands via the [portal](https://www.broloket.nl/ondergrondgegevens) of the Basis Registratie Ondergrond (**BRO**). GeoST can easily load
this data for an area of interest.

In [ ]:
import geost

# Read a few BRO pedological soil cores in a small area 250 m x 500 m
boreholes = geost.bro_api_read("BHR-P", bbox=(142_000, 455_000, 142_250, 455_500))
boreholes

Collection:
  header (rows, columns): (8, 11)
  data (rows, columns): (40, 8)
crs: EPSG:28992
# surveys = 8

You can see that this directly loads the soil cores into a `geost.Collection`. This is
also supported for geological (BHR-G) and geotechnical (BHR-GT) boreholes, cone penetration
test (CPT) data and pedological soil profile descriptions (SFR). This facilitates the
direct use of BRO data within any application.

### Using local files
A likely option is to use GeoST to load survey data stored in a tabular format such as
Parquet or csv and use the available selection and analysis methods. For example, suppose you
have survey data for multiple boreholes stored in a local Parquet file. Using
[`geost.read_borehole_table`](../api_reference/generated/geost.read_borehole_table.rst) you can
very easily load it into a `Collection` or if preferred, a `pandas.DataFrame` and use the data
for further analysis:

In [ ]:
borehole_file = geost.data.boreholes_usp(
    return_filepath=True
)  # Use the filepath instead of directly reading the borehole data
boreholes = geost.read_borehole_table(borehole_file)
print(boreholes)

Collection:
  header (rows, columns): (67, 5)
  data (rows, columns): (1398, 32)
crs: None
# surveys = 67



C:\Users\knaake\AppData\Local\Temp\ipykernel_17220\838550156.py:4: DeprecationWarning: The function read_borehole_table is deprecated and will be removed in a future version. Please use read_table instead.
  boreholes = geost.read_borehole_table(borehole_file)


Now you can easily select the boreholes that contain peat ("V") for example:

In [7]:
peat_boreholes = boreholes.select_by_values("lith", "V")
peat_boreholes

Collection:
  header (rows, columns): (32, 5)
  data (rows, columns): (670, 32)
crs: None
# surveys = 32

The same thing is possible too for Pandas DataFrames. Note that with DataFrames, any GeoST
methods need to be used through the `.gst` accessor, like we showed in the [Data structures](./data_structures.ipynb#geost-accessor) section. This way we can select the boreholes
that contain peat just like before. Let's load the same borehole data, but this time as a
`pandas.DataFrame`:

In [8]:
borehole_df = geost.read_borehole_table(borehole_file, as_collection=False)
borehole_df.head()

C:\Users\knaake\AppData\Local\Temp\ipykernel_17220\1583501152.py:1: DeprecationWarning: The function read_borehole_table is deprecated and will be removed in a future version. Please use read_table instead.
  borehole_df = geost.read_borehole_table(borehole_file, as_collection=False)


,nr,x,y,surface,end,top,bottom,lith,zm,zmk,...,cons,color,lutum_pct,plants,shells,kleibrokjes,strat_1975,strat_2003,strat_inter,desc
0,B31H0541,139585,456000,1.2,-9.9,0.00,0.20,K,NaN,NaN,...,NaN,ON,NaN,0,0,0,NaN,EC,NaN,[TEELAARDE#***#****#*] ..........................
1,B31H0541,139585,456000,1.2,-9.9,0.20,0.60,K,NaN,NaN,...,NaN,BR,NaN,0,0,0,NaN,EC,NaN,[KLEI#***#****#*] grysbruin.
2,B31H0541,139585,456000,1.2,-9.9,0.60,0.95,V,NaN,NaN,...,NaN,BR,NaN,0,0,0,NaN,NI,NaN,[VEEN#***#****#*] donkerbruin.
3,B31H0541,139585,456000,1.2,-9.9,0.95,2.80,Z,NaN,ZMFO,...,NaN,GR,NaN,0,0,0,NaN,EC,NaN,[ZAND#***#****#*] FYN TOT matig fyn# iets slib...
4,B31H0541,139585,456000,1.2,-9.9,2.80,4.20,Z,NaN,ZFC,...,NaN,BR,NaN,0,0,0,NaN,BXWI,NaN,[ZAND#***#****#*] fyn# grysbruin.


Now we can make the same selection using the `.gst` accessor:

In [9]:
peat_df = borehole_df.gst.select_by_values("lith", "V")
peat_df

,nr,x,y,surface,end,top,bottom,lith,zm,zmk,...,cons,color,lutum_pct,plants,shells,kleibrokjes,strat_1975,strat_2003,strat_inter,desc
0,B31H0541,139585,456000,1.20,-9.90,0.00,0.20,K,NaN,NaN,...,NaN,ON,NaN,0,0,0,NaN,EC,NaN,[TEELAARDE#***#****#*] ..........................
1,B31H0541,139585,456000,1.20,-9.90,0.20,0.60,K,NaN,NaN,...,NaN,BR,NaN,0,0,0,NaN,EC,NaN,[KLEI#***#****#*] grysbruin.
2,B31H0541,139585,456000,1.20,-9.90,0.60,0.95,V,NaN,NaN,...,NaN,BR,NaN,0,0,0,NaN,NI,NaN,[VEEN#***#****#*] donkerbruin.
3,B31H0541,139585,456000,1.20,-9.90,0.95,2.80,Z,NaN,ZMFO,...,NaN,GR,NaN,0,0,0,NaN,EC,NaN,[ZAND#***#****#*] FYN TOT matig fyn# iets slib...
4,B31H0541,139585,456000,1.20,-9.90,2.80,4.20,Z,NaN,ZFC,...,NaN,BR,NaN,0,0,0,NaN,BXWI,NaN,[ZAND#***#****#*] fyn# grysbruin.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1352,B32C1800,140571,455934,2.08,-2.92,0.70,1.20,Z,NaN,ZMF,...,NaN,GR,NaN,0,0,0,NaN,NaN,NaN,BRON:Boormanager;VAN:70.0000;TOT:120.0000;H:Z;...
1353,B32C1800,140571,455934,2.08,-2.92,1.20,1.50,K,NaN,NaN,...,NaN,GR,NaN,0,0,0,NaN,NaN,NaN,BRON:Boormanager;VAN:120.0000;TOT:150.0000;H:K...
1354,B32C1800,140571,455934,2.08,-2.92,1.50,1.70,V,NaN,NaN,...,NaN,BR,NaN,0,0,0,NaN,NaN,NaN,BRON:Boormanager;VAN:150.0000;TOT:170.0000;H:V...
1355,B32C1800,140571,455934,2.08,-2.92,1.70,2.00,Z,NaN,ZUF,...,NaN,BR,NaN,0,0,0,NaN,NaN,NaN,BRON:Boormanager;VAN:170.0000;TOT:200.0000;H:Z...


Notice that the shape of `peat_df` is the same as before in the data table of the `peat_boreholes` Collection: 670 rows x 32 columns.




## Reading model data
Model data comprise 3D voxel- and layermodels (such as GeoTOP, REGIS II) or geological maps.

| File format/data service | Read function  | Returned GeoST object | Description  |
| ------------------------ | -------------- | --------------------- | -----------  |  
| Generic Voxelmodel | [`VoxelModel`](../api_reference/generated/geost.models.VoxelModel.rst) | [`VoxelModel`](../api_reference/voxelmodel.rst) | Generic reader for a voxelmodel presented in NetCDF format (readable as xarray dataset) |
| Generic Voxelmodel OpenDAP | [`VoxelModel.from_opendap`](../api_reference/generated/geost.models.VoxelModel.from_opendap.rst) | [`VoxelModel`](../api_reference/voxelmodel.rst) | Generic reader for a voxelmodel from an OpenDAP server |
| GeoTOP NetCDF | [`GeoTop.from_netcdf`](../api_reference/generated/geost.bro.bro_geotop.GeoTop.from_netcdf.rst) | [`VoxelModel`](../api_reference/voxelmodel.rst) | Reader for GeoTOP NetCDF distribution |
| GeoTOP OpenDAP| [`GeoTop.from_opendap`](../api_reference/generated/geost.bro.bro_geotop.GeoTop.from_opendap.rst) | [`VoxelModel`](../api_reference/voxelmodel.rst) | Reader for GeoTOP OpenDAP distribution |

## Data reading examples

In [ ]:
print(type(boreholes))
boreholes.header

: 

: 

In [ ]:
from geost.bro import GeoTop

# Get corresponding voxels of the GeoTOP model
geotop = GeoTop.from_opendap(bbox=(142_000, 455_000, 142_250, 455_500))

geotop